# WTA Match Predictor - Model Training

## Contents

- Import Data and Packages
- Preprocessing and Defining Features
- Train, Test, and Evaluate Models

### Import Data and Required Packages

In [2]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns # for scrambling columns
import random
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [3]:
df = pd.read_csv('data/stud.csv')
df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2023-609,Indian Wells,Hard,96,PM,20230306,286,214544,2.0,NaN,...,58.0,37.0,16.0,13.0,8.0,11.0,2,6100,16,2205
1,2023-609,Indian Wells,Hard,96,PM,20230306,285,216347,1.0,NaN,...,55.0,30.0,4.0,10.0,3.0,8.0,1,10585,36,1303
2,2023-609,Indian Wells,Hard,96,PM,20230306,284,221054,NaN,NaN,...,52.0,37.0,10.0,12.0,5.0,8.0,77,784,13,2246
3,2023-609,Indian Wells,Hard,96,PM,20230306,283,201514,NaN,NaN,...,24.0,14.0,12.0,8.0,0.0,5.0,83,743,43,1200
4,2023-609,Indian Wells,Hard,96,PM,20230306,282,201614,5.0,NaN,...,50.0,34.0,21.0,14.0,1.0,4.0,5,4905,49,1080


### Preprocessing and Defining Features

In [4]:
# First, we want to drop columns that we know will not be useful for our model or understanding.
# Information related to the tournament is not relevant
df.drop(columns=['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date'], axis=1, inplace=True)
# Certain information about players (e.g. height, age, hand) are also irrelevant
df.drop(columns=['winner_hand', 'loser_hand', 'winner_ht', 'loser_ht', 'winner_age', 'loser_age', 'winner_ioc', 'loser_ioc'], axis=1, inplace=True)
# Certain match information will also be irrelevant for predictions
df.drop(columns=['match_num', 'best_of', 'round', 'minutes', 'score'], axis=1, inplace=True)
# For now, we will also drop most columns outside of statistics, although we may use some of these in future iterations
df.drop(columns=['winner_seed', 'loser_seed', 'winner_entry', 'loser_entry', 'winner_rank', 'loser_rank', 'winner_rank_points', 'loser_rank_points'], axis=1, inplace=True)
# We will also drop 'bp_saved' for both players, since we found this was not a meaningful statistic in our EDA
df.drop(columns=['w_bpSaved', 'l_bpSaved'], axis=1, inplace=True)

In [5]:
df.head()

,winner_id,winner_name,loser_id,loser_name,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpFaced
0,214544,Aryna Sabalenka,206252,Barbora Krejcikova,11.0,4.0,84.0,53.0,39.0,12.0,14.0,4.0,3.0,6.0,88.0,58.0,37.0,16.0,13.0,11.0
1,216347,Iga Swiatek,215370,Bianca Andreescu,1.0,0.0,73.0,50.0,31.0,10.0,11.0,8.0,1.0,2.0,72.0,55.0,30.0,4.0,10.0,8.0
2,221054,Emma Raducanu,206242,Beatriz Haddad Maia,2.0,4.0,83.0,55.0,38.0,14.0,13.0,5.0,4.0,4.0,76.0,52.0,37.0,10.0,12.0,8.0
3,201514,Sorana Cirstea,211095,Bernarda Pera,4.0,0.0,44.0,22.0,19.0,12.0,8.0,1.0,3.0,5.0,55.0,24.0,14.0,12.0,8.0,5.0
4,201614,Caroline Garcia,220367,Leylah Fernandez,11.0,3.0,93.0,49.0,42.0,26.0,15.0,2.0,5.0,8.0,91.0,50.0,34.0,21.0,14.0,4.0


Something very important to note is that, if we implicitly define matches as already having a winner and loser, our model will always predict the match outcome with 100% accuracy, since the data will always indicate who wins and who loses. To fix this, we need to scramble the columns for winners and losers, so it's not obvious to the model who wins the match. We will also rename columns so that there is no mention of a winner or a loser, and will instead encode winner/loser as a single number, 0 (for player 1) or 1 (for player 2).

In [6]:
# First: Rename 'winner_'/'w_' and 'loser_'/'l_' columns to represent player 1 and player 2 respectively
for col in df.columns:
    if 'winner' in col:
        df.rename(columns={col: col.replace('winner', 'p1')}, inplace=True)
    elif 'w' in col:
        df.rename(columns={col: col.replace('w', 'p1')}, inplace=True)
    elif 'loser' in col:
        df.rename(columns={col: col.replace('loser', 'p2')}, inplace=True)
    elif 'l' in col:
        df.rename(columns={col: col.replace('l', 'p2')}, inplace=True)
df['match_winner'] = 0 # create match winner column with all 0s (player 1 for all obs.)

In [7]:
df.head()

,p1_id,p1_name,p2_id,p2_name,p1_ace,p1_df,p1_svpt,p1_1stIn,p1_1stWon,p1_2ndWon,...,p1_bpFaced,p2_ace,p2_df,p2_svpt,p2_1stIn,p2_1stWon,p2_2ndWon,p2_SvGms,p2_bpFaced,match_winner
0,214544,Aryna Sabalenka,206252,Barbora Krejcikova,11.0,4.0,84.0,53.0,39.0,12.0,...,4.0,3.0,6.0,88.0,58.0,37.0,16.0,13.0,11.0,0
1,216347,Iga Swiatek,215370,Bianca Andreescu,1.0,0.0,73.0,50.0,31.0,10.0,...,8.0,1.0,2.0,72.0,55.0,30.0,4.0,10.0,8.0,0
2,221054,Emma Raducanu,206242,Beatriz Haddad Maia,2.0,4.0,83.0,55.0,38.0,14.0,...,5.0,4.0,4.0,76.0,52.0,37.0,10.0,12.0,8.0,0
3,201514,Sorana Cirstea,211095,Bernarda Pera,4.0,0.0,44.0,22.0,19.0,12.0,...,1.0,3.0,5.0,55.0,24.0,14.0,12.0,8.0,5.0,0
4,201614,Caroline Garcia,220367,Leylah Fernandez,11.0,3.0,93.0,49.0,42.0,26.0,...,2.0,5.0,8.0,91.0,50.0,34.0,21.0,14.0,4.0,0


In [8]:
# Second: Randomly shuffle p1 and p2 columns to ensure no pattern
cols1 = [c for c in df.columns if 'p1' in c]
cols2 = [c for c in df.columns if 'p2' in c]
cols2_targ = [c.replace('p1', 'p2') for c in cols1]
print(cols1)
print(cols2)

# copy df for scrambling
copy = df.copy()

#maskIdx = [i for i in df.index if i % 3]
# generate len(df.index) / 2 random indices. for those observations, swap columns and set match_winner to 1
maskIdx = [random.randint(0, len(df.index) - 1) for i in range((int)(len(df.index) / 2))]
print(maskIdx)
df.loc[maskIdx, cols1] = copy.loc[maskIdx, cols2_targ].values
df.loc[maskIdx, cols2_targ] = copy.loc[maskIdx, cols1].values
df.loc[maskIdx, 'match_winner'] = 1

['p1_id', 'p1_name', 'p1_ace', 'p1_df', 'p1_svpt', 'p1_1stIn', 'p1_1stWon', 'p1_2ndWon', 'p1_SvGms', 'p1_bpFaced']
['p2_id', 'p2_name', 'p2_ace', 'p2_df', 'p2_svpt', 'p2_1stIn', 'p2_1stWon', 'p2_2ndWon', 'p2_SvGms', 'p2_bpFaced']
[72, 78, 106, 97, 58, 60, 21, 43, 57, 70, 73, 6, 94, 97, 60, 78, 64, 65, 21, 45, 67, 109, 110, 16, 39, 69, 106, 61, 28, 103, 85, 24, 92, 44, 47, 29, 43, 105, 14, 62, 98, 6, 40, 32, 79, 71, 56, 99, 78, 39, 97, 59, 35, 7, 74, 40]


In [9]:
# By running all the cells in this notebook, you can see that the columns are randomly shuffled, indicated by the match winner changing between 0 and 1 for different observations.
df.head(10)

,p1_id,p1_name,p2_id,p2_name,p1_ace,p1_df,p1_svpt,p1_1stIn,p1_1stWon,p1_2ndWon,...,p1_bpFaced,p2_ace,p2_df,p2_svpt,p2_1stIn,p2_1stWon,p2_2ndWon,p2_SvGms,p2_bpFaced,match_winner
0,214544,Aryna Sabalenka,206252,Barbora Krejcikova,11.0,4.0,84.0,53.0,39.0,12.0,...,4.0,3.0,6.0,88.0,58.0,37.0,16.0,13.0,11.0,0
1,216347,Iga Swiatek,215370,Bianca Andreescu,1.0,0.0,73.0,50.0,31.0,10.0,...,8.0,1.0,2.0,72.0,55.0,30.0,4.0,10.0,8.0,0
2,221054,Emma Raducanu,206242,Beatriz Haddad Maia,2.0,4.0,83.0,55.0,38.0,14.0,...,5.0,4.0,4.0,76.0,52.0,37.0,10.0,12.0,8.0,0
3,201514,Sorana Cirstea,211095,Bernarda Pera,4.0,0.0,44.0,22.0,19.0,12.0,...,1.0,3.0,5.0,55.0,24.0,14.0,12.0,8.0,5.0,0
4,201614,Caroline Garcia,220367,Leylah Fernandez,11.0,3.0,93.0,49.0,42.0,26.0,...,2.0,5.0,8.0,91.0,50.0,34.0,21.0,14.0,4.0,0
5,214954,Marketa Vondrousova,202460,Ons Jabeur,0.0,5.0,68.0,37.0,23.0,17.0,...,4.0,2.0,2.0,71.0,37.0,25.0,10.0,11.0,8.0,0
6,203354,Martina Trevisan,214096,Karolina Muchova,0.0,4.0,97.0,65.0,41.0,17.0,...,13.0,7.0,7.0,97.0,58.0,43.0,19.0,15.0,8.0,1
7,211651,Paula Badosa,214981,Elena Rybakina,2.0,7.0,63.0,34.0,23.0,13.0,...,8.0,9.0,1.0,76.0,44.0,30.0,19.0,10.0,6.0,1
8,201514,Sorana Cirstea,214096,Karolina Muchova,4.0,1.0,53.0,39.0,26.0,9.0,...,4.0,3.0,1.0,63.0,40.0,20.0,10.0,9.0,8.0,0
9,214954,Marketa Vondrousova,201662,Karolina Pliskova,4.0,3.0,46.0,31.0,23.0,6.0,...,6.0,2.0,5.0,37.0,26.0,8.0,3.0,7.0,8.0,0


Now that our data is sufficiently preprocessed, we can define our X and y features. `X` will contain all of our input features, while `y` will be our single target output.

In [10]:
# Define X: all columns except p1/p2 name (irrelevant to model) and the match winner
X = df.drop(columns=['p1_name', 'p2_name', 'match_winner'], axis=1)
# Define y: match winner
y = df['match_winner']

In [11]:
X.head()

,p1_id,p2_id,p1_ace,p1_df,p1_svpt,p1_1stIn,p1_1stWon,p1_2ndWon,p1_SvGms,p1_bpFaced,p2_ace,p2_df,p2_svpt,p2_1stIn,p2_1stWon,p2_2ndWon,p2_SvGms,p2_bpFaced
0,214544,206252,11.0,4.0,84.0,53.0,39.0,12.0,14.0,4.0,3.0,6.0,88.0,58.0,37.0,16.0,13.0,11.0
1,216347,215370,1.0,0.0,73.0,50.0,31.0,10.0,11.0,8.0,1.0,2.0,72.0,55.0,30.0,4.0,10.0,8.0
2,221054,206242,2.0,4.0,83.0,55.0,38.0,14.0,13.0,5.0,4.0,4.0,76.0,52.0,37.0,10.0,12.0,8.0
3,201514,211095,4.0,0.0,44.0,22.0,19.0,12.0,8.0,1.0,3.0,5.0,55.0,24.0,14.0,12.0,8.0,5.0
4,201614,220367,11.0,3.0,93.0,49.0,42.0,26.0,15.0,2.0,5.0,8.0,91.0,50.0,34.0,21.0,14.0,4.0


Since all of our predictive features are already quantitative, we don't need to worry about encoding any categorical variables. In this case, we can move directly onto modelling.

### Train, Test, and Evaluate Models

In [12]:
# Separate dataset into training and testing sets (80/20 split)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((90, 18), (23, 18))

In [ ]:
# Define model evaluation function
# Use built-in functions to find most relevant statistics for model strength and accuracy
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, mse, rmse, r2_square